In [ ]:
# ==============================================================================
# CELL 1: Setup Environment and Mount Drive
# ==============================================================================
!pip install -q transformers datasets evaluate accelerate

from google.colab import drive
import os

drive.mount('/content/drive')

# Define paths
PROJECT_DIR = '/content/drive/MyDrive/Smart_Scan/Recognition_Model'
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'trocr_checkpoints')
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, 'trocr_final')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

print(f"✅ Environment ready. Saving checkpoints to: {CHECKPOINT_DIR}")

In [ ]:
# ==============================================================================
# CELL 2: Load and Prepare the Dataset
# ==============================================================================
from datasets import load_dataset
from transformers import TrOCRProcessor

print("📥 Downloading Im2LaTeX dataset from Hugging Face...")
try:
    # Using the standard im2latex-100k dataset and correct split names
    train_data = load_dataset("yuntian-deng/im2latex-100k", split="train[:100000]")
    eval_data = load_dataset("yuntian-deng/im2latex-100k", split="val[:5000]") # Changed to 'val'
    print(f"✅ Dataset loaded: {len(train_data)} training samples, {len(eval_data)} validation samples")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    raise

# Load processor
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-printed")

def preprocess_data(examples):
    # Convert images to tensors
    pixel_values = processor(examples["image"], return_tensors="pt").pixel_values
    # Tokenize the LaTeX formulas
    labels = processor.tokenizer(
        examples["formula"], padding="max_length", max_length=64
    ).input_ids
    # Ignore padding in loss calculation
    labels = [
        [tok if tok != processor.tokenizer.pad_token_id else -100 for tok in seq]
        for seq in labels
    ]
    return {"pixel_values": pixel_values.squeeze(), "labels": labels}

print("⚙️ Processing images and LaTeX tokens... (This takes a few minutes)")
train_dataset = train_data.map(preprocess_data, remove_columns=["image", "formula"], batched=True)
eval_dataset = eval_data.map(preprocess_data, remove_columns=["image", "formula"], batched=True)

print(f"✅ Preprocessing complete! Train: {len(train_dataset)} samples, Eval: {len(eval_dataset)} samples")


In [ ]:
# ==============================================================================
# CELL 3: Initialize Model Architecture
# ==============================================================================
from transformers import VisionEncoderDecoderModel

print("🧠 Building Vision-Encoder-Decoder Model...")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-printed")

# Configure LaTeX specific decoding rules
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 64
model.config.early_stopping = True
model.config.num_beams = 4  # Improves LaTeX accuracy

## ⏱️ Training Times & GPU Pricing (TrOCR Recognition)

### Dataset Size Comparison (A100 40GB):

| Dataset Size | Per Epoch | 5 Epochs Total | Cost ($0.54/hr) | Notes |
|-------------|----------|----------------|----------------|-------|
| **5K** ✅ | 18-24 min | **90-120 min** | **$0.81-$1.08** | **RECOMMENDED** |
| **20K** | 72-96 min | **6-8 hours** | **$3.24-$4.32** | Good accuracy |
| **50K** | 180-240 min | **15-20 hours** | **$8.10-$10.80** | Very high accuracy |
| **100K** | 360-480 min | **30-40 hours** | **$16.20-$21.60** | Maximum accuracy |

### 💰 With $10 Budget:

| Dataset Size | Cost per Training | Trainings with $10 |
|-------------|------------------|-------------------|
| **5K** ✅ | $0.81-$1.08 | **9+ times** |
| **20K** | $3.24-$4.32 | **2-3 times** |
| **50K** | $8.10-$10.80 | **1 time** |
| **100K** | $16.20-$21.60 | ❌ Over budget |

### 🎯 Recommendations:
- ✅ **5K samples** (current): Fastest, cheapest, good accuracy
- ✅ **20K samples**: Better accuracy, still affordable
- ⚠️ **50K samples**: Very high accuracy, nearly hits $10 limit
- ❌ **100K samples**: Too expensive for $10 budget

**Change in notebook:**
```python
# Line to change - from:
train_data = load_dataset("im2latex", split="train[:5000]")
eval_data = load_dataset("im2latex", split="validation[:500]")

# To (20K example):
train_data = load_dataset("im2latex", split="train[:20000]")
eval_data = load_dataset("im2latex", split="validation[:2000]")

# Or (50K):
train_data = load_dataset("im2latex", split="train[:50000]")
eval_data = load_dataset("im2latex", split="validation[:5000]")
```


In [ ]:
# ==============================================================================
# CELL 4: The Auto-Resume Training Loop (Colab Pro Resilient)
# ==============================================================================
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers.trainer_utils import get_last_checkpoint

# --- A100 OPTIMIZED TRAINING ARGUMENTS ---
training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    predict_with_generate=True,
    eval_strategy="steps",          
    generation_max_length=64,   
    per_device_train_batch_size=64,     # MAXIMIZES A100 VRAM
    per_device_eval_batch_size=64,      
    dataloader_num_workers=16,          # FEEDS DATA FASTER
    fp16=True,                          # GPU ACCELERATION
    logging_steps=50,
    save_steps=500,                     
    eval_steps=500,
    save_total_limit=3,
    num_train_epochs=5,                 # 5 EPOCHS IS PLENTY FOR 100K DATA
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# --- AUTO-RESUME LOGIC FOR COLAB PRO ---
last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

try:
    if last_checkpoint is not None:
        print(f"🔄 Found interrupted training at {last_checkpoint}. Resuming...")
        trainer.train(resume_from_checkpoint=last_checkpoint)
        print("✅ Training resumed successfully!")
    else:
        print("🚀 Starting fresh training session...")
        trainer.train()
        print("✅ Training completed successfully!")
except KeyboardInterrupt:
    print("⏸️ Training paused. Run this cell again to resume from last checkpoint.")
except Exception as e:
    print(f"❌ Training error: {e}")
    print("Run this cell again to resume from last checkpoint.")
    raise


In [ ]:
# ==============================================================================
# CELL 5: Save Final Model
# ==============================================================================
print("💾 Saving final compiled model to Drive...")
trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)
print(f"🎉 Recognition Model Complete! Files saved in: {FINAL_MODEL_DIR}")

In [ ]:
# ==============================================================================
# CELL 6: Verify Model Save and List Drive Contents
# ==============================================================================
import os

print("🔍 Verifying saved files...")

# Check if final model exists
final_model_files = os.listdir(FINAL_MODEL_DIR) if os.path.exists(FINAL_MODEL_DIR) else []
print(f"\n📁 Final Model Directory: {FINAL_MODEL_DIR}")
print(f"   Files: {final_model_files}")

# Check checkpoints
checkpoint_files = os.listdir(CHECKPOINT_DIR) if os.path.exists(CHECKPOINT_DIR) else []
print(f"\n📁 Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"   Files: {checkpoint_files[:5]}...")  # Show first 5

if final_model_files:
    print("\n✅ Model saved successfully on Drive!")
else:
    print("\n⚠️ Warning: No files found in final model directory")
